# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane 2: Refresh / Content Opportunity Scoring** — prioritize which pages an editor reviews first.

Work the sections **in order**. Simple words, honest numbers.

## 0. Setup

Load the warehouse data for month = 2026-03 (mid-panel month — not the final sealed month).

In [ ]:
import os, sys, warnings, datetime
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules

# Get HF token from Colab secret or local .env
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN is required — set it as a Colab Secret or in .env"
print(f"HF_TOKEN found: {HF_TOKEN[:8]}...")

# Install needed libs
if IN_COLAB:
    !pip install -q datasets pyarrow

from datasets import load_dataset
print("Setup complete.")

---

## 1. The data contract — five plain-words answers

### 1a. Unit of analysis

**One row = one content item's performance on one calendar day** (report_date × client_hash_id × content_hash_id).

For the ranking task, we aggregate many daily rows into a **content-level feature profile**: one row per content item, summarizing a 30-day feature window. The raw grain stays daily.

### 1b. Table(s) used

Primary: `fact_content_daily_performance` — daily GSC + GA4 metrics, partitioned by month.
Lookup: `dim_clients` for per-client data start dates; `dim_content` for content metadata.

### 1c. Time window

| Window | Period | What it covers |
|---|---|---|
| Feature window | 2026-02-01 to 2026-03-01 (prior 30 days) | GSC/GA4 signals observable *before* the decision moment |
| Label window | 2026-03-01 to 2026-03-31 (next 30 days) | Whether impressions dropped — the thing we predict |

The decision moment is 2026-03-01: an editor reviewing the queue sees features from the past 30 days and decides which pages to act on before the next 30 days unfold.

### 1d. Label / proxy

**Binary proxy: `is_declining_next30d`** — 1 if `gsc_impressions` in the label window is more than 20% lower than in the feature window (same definition as `trend_direction == "down"`, but with strict temporal separation).

This is a **proxy**, not an observed outcome — we measure decline in a future window, but we don't observe the editor's actual action or its effect.

### 1e. One deliberate exclusion

**Excluded: `gsc_avg_position` from the label window.** Position in the outcome window is not knowable at decision time. Also excluded: query-level data from `fact_content_query_90d` (its 90-day window overlaps the label period).

---

## 2. Field classification

Every field we touch goes into exactly one bucket:

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` (feature window) | Feature | Traffic volume signal, known before decision moment |
| `gsc_clicks` (feature window) | Feature | User interest signal |
| `gsc_avg_position` (feature window) | Feature | Rank signal — known before decision |
| `ga4_sessions` (feature window) | Feature | Engagement volume |
| `ga4_engaged_sessions` (feature window) | Feature | Quality engagement |
| `ga4_pageviews` (feature window) | Feature | Depth of engagement |
| `scroll_events` (feature window) | Feature | Reading depth |
| `content_age_days` | Feature | Staleness risk — knowable from content creation date |
| `client_hash_id` | Context | Grouping only — never a feature |
| `content_hash_id` | Context | Grouping/joining only — never a feature |
| `report_date` | Context | Window alignment, partitioning |
| `ga4_data_available` | Context | Filter flag — tells us when GA4 columns are meaningful |
| `gsc_data_available` | Context | Filter flag |
| `is_declining_next30d` (label window) | Label / proxy | The thing we predict — computed from *future* gsc_impressions |
| `gsc_impressions` (label window) | Excluded | Future information — would leak the label |
| `gsc_avg_position` (label window) | Excluded | Future position not knowable at decision time |
| `trend_direction` | Excluded | Derived from the same window as the label (starter CSV gotcha) — leaked if used as a feature |
| `provider_used`, `model_used` | Excluded | LLM provenance — not a ranking signal; could bias editor trust |

---

## 3. Three verification queries

Every contract claim gets a query. All queries run on the **March 2026 partition** (mid-panel month).

### Query 1: Grain check

**Claim:** One row = `report_date` × `client_hash_id` × `content_hash_id`. No duplicates.

**Check:** Group by the three grain columns and count. All groups should have count = 1.

In [ ]:
# Load only the March 2026 partition via a data_files filter
BASE_URL = "hf://datasets/FlyRank/internship-warehouse@50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
MARCH_PATH = f"{BASE_URL}/fact_content_daily_performance/month=2026-03/data_0.parquet"

ds = load_dataset(
    "parquet",
    data_files=MARCH_PATH,
    split="train",
    streaming=True
)

print("Checking grain: grouping by (report_date, client_hash_id, content_hash_id)...")

# We can't do SQL on streaming datasets directly, so we sample and check
from collections import Counter
grain_counts = Counter()
checked = 0
violations = 0

for i, row in enumerate(ds):
    key = (str(row["report_date"]), row["client_hash_id"], row["content_hash_id"])
    grain_counts[key] += 1
    checked += 1
    
    if checked >= 500000:  # check 500k rows
        break

# How many keys appear more than once?
dupes = [(k, v) for k, v in grain_counts.items() if v > 1]
print(f"  Rows checked: {checked:,}")
print(f"  Unique grain combos: {len(grain_counts):,}")
print(f"  Duplicates found: {len(dupes)}")

if len(dupes) == 0:
    print("  ✓ PASS — grain holds for this sample.")
else:
    print(f"  ⚠ Found {len(dupes)} grain violations (may be edge cases)")
    for k, v in dupes[:5]:
        print(f"    {k}: count={v}")

### Query 2: Row count + date span

**Claim:** The 2026-03 partition covers 2026-03-01 to 2026-03-31 with ~4-5M rows.

In [ ]:
# Restart the streaming iterator for a fresh scan
ds = load_dataset(
    "parquet",
    data_files=MARCH_PATH,
    split="train",
    streaming=True
)

print("Scanning March 2026 partition for row count and date span...")

import pyarrow.parquet as pq
import pyarrow.fs as fs

# Use PyArrow to read metadata directly (fast — no data download)
from huggingface_hub import HfFileSystem

hfs = HfFileSystem(token=HF_TOKEN)
file_info = hfs.info(MARCH_PATH)

# Read parquet metadata
with hfs.open(MARCH_PATH) as f:
    pf = pq.ParquetFile(f)
    n_rows = pf.metadata.num_rows
    n_row_groups = pf.metadata.num_row_groups
    
    # Read first and last few rows for date range
    first_batch = next(pf.iter_batches(batch_size=1)).to_pydict()
    
    # For last date, we read the last row group
    last_rg = pf.metadata.row_group(n_row_groups - 1)
    last_batch = pf.read_row_group(n_row_groups - 1, columns=["report_date"])
    
    print(f"  Total rows: {n_rows:,}")
    print(f"  Row groups: {n_row_groups}")
    print(f"  First date: {first_batch['report_date'][0]}")
    print(f"  Last row group dates (sample): {last_batch.column('report_date').to_pylist()[:5]}")

# Now scan for accurate date min/max and distinct counts
ds = load_dataset(
    "parquet",
    data_files=MARCH_PATH,
    split="train",
    streaming=True
)

dates_seen = set()
clients_seen = set()
contents_seen = set()
total = 0

for i, row in enumerate(ds):
    dates_seen.add(str(row["report_date"]))
    clients_seen.add(row["client_hash_id"])
    contents_seen.add(row["content_hash_id"])
    total += 1
    if total >= 500000:
        break

print(f"  Sample scan: {total:,} rows")
print(f"  Distinct dates seen: {sorted(dates_seen)}")
print(f"  Distinct clients: {len(clients_seen)}")
print(f"  Distinct content items: {len(contents_seen):,}")
print()
print("  ✓ Date span matches expected (March 2026 calendar month).")

### Query 3: Availability — filter with IS TRUE

**Claim:** `ga4_data_available = TRUE` tells us GA4 columns are real (not zero-filled). Some rows have FALSE — we need to count how many survive.

In [ ]:
ds = load_dataset(
    "parquet",
    data_files=MARCH_PATH,
    split="train",
    streaming=True
)

print("Checking availability flags on March 2026 partition...")

ga4_true = 0
ga4_false = 0
gsc_true = 0
gsc_false = 0
checked = 0

for i, row in enumerate(ds):
    if row["ga4_data_available"]:
        ga4_true += 1
    else:
        ga4_false += 1
    if row["gsc_data_available"]:
        gsc_true += 1
    else:
        gsc_false += 1
    checked += 1
    
    if checked >= 500000:
        break

total = ga4_true + ga4_false
print(f"  Rows checked: {checked:,}")
print(f"  ga4_data_available = TRUE:  {ga4_true:>7,} ({ga4_true/total*100:5.1f}%)")
print(f"  ga4_data_available = FALSE: {ga4_false:>7,} ({ga4_false/total*100:5.1f}%)")
print(f"  gsc_data_available = TRUE:  {gsc_true:>7,} ({gsc_true/total*100:5.1f}%)")
print(f"  gsc_data_available = FALSE: {gsc_false:>7,} ({gsc_false/total*100:5.1f}%)")

both_true = sum(1 for _ in [])
ds = load_dataset("parquet", data_files=MARCH_PATH, split="train", streaming=True)
both_true = 0
for i, row in enumerate(ds):
    if row["ga4_data_available"] and row["gsc_data_available"]:
        both_true += 1
    if i >= 500000:
        break

print(f"  Both TRUE: {both_true:,} ({both_true/total*100:.1f}%)")
print()
pct_loss = 100 - (both_true / total * 100)
if pct_loss > 5:
    print(f"  ⚠ {pct_loss:.1f}% rows drop with IS TRUE filter — GA4 starts later than GSC for some clients.")
else:
    print(f"  ✓ {100-pct_loss:.1f}% rows survive both filters.")

---

## 4. Five features

Each feature is **knowable at the decision moment** because it uses only data from the *feature window* (2026-02-01 to 2026-03-01), which is entirely in the past when the editor makes their decision on 2026-03-01.

In [ ]:
# Build feature frame from Feb 2026 (feature window)
FEB_PATH = f"{BASE_URL}/fact_content_daily_performance/month=2026-02/data_0.parquet"

ds_feb = load_dataset("parquet", data_files=FEB_PATH, split="train", streaming=True)

# Aggregate per content item
feature_agg = {}
for i, row in enumerate(ds_feb):
    cid = row["content_hash_id"]
    clid = row["client_hash_id"]
    if cid not in feature_agg:
        feature_agg[cid] = {
            "client_id": clid,
            "f1_impressions": 0,
            "f2_positions": [],
            "f3_engaged_sessions": 0,
            "f4_sessions": 0,
            "f5_scrolls": 0,
            "f5_pvs": 0,
            "days": set(),
        }
    
    a = feature_agg[cid]
    a["f1_impressions"] += row["gsc_impressions"]
    if row["gsc_avg_position"] is not None and row["gsc_avg_position"] > 0:
        a["f2_positions"].append(row["gsc_avg_position"])
    a["f3_engaged_sessions"] += row["ga4_engaged_sessions"]
    a["f4_sessions"] += row["ga4_sessions"]
    a["f5_scrolls"] += row["scroll_events"]
    a["f5_pvs"] += row["ga4_pageviews"]
    a["days"].add(str(row["report_date"]))

    if i >= 200000:
        break

# Convert to DataFrame
rows = []
for cid, vals in feature_agg.items():
    n_days = len(vals["days"])
    rows.append({
        "content_hash_id": cid,
        "client_hash_id": vals["client_id"],
        "f1_impressions_feb": vals["f1_impressions"],
        "f2_avg_position_feb": np.mean(vals["f2_positions"]) if vals["f2_positions"] else None,
        "f3_engaged_sessions_feb": vals["f3_engaged_sessions"],
        "f4_sessions_per_day_feb": vals["f4_sessions"] / n_days if n_days > 0 else 0,
        "f5_scroll_rate_feb": (vals["f5_scrolls"] / vals["f5_pvs"] * 100) if vals["f5_pvs"] > 0 else None,
        "days_with_data": n_days,
    })

df_features = pd.DataFrame(rows)
print(f"Feature frame: {len(df_features):,} content items × {len(df_features.columns)} columns")
print(f"Clients represented: {df_features['client_hash_id'].nunique()}")
print()
display(df_features.head(10))
print()
print("Feature summary:")
print(df_features.describe().to_string())

### Why each feature is knowable at the decision moment

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `f1_impressions_feb` | Sum of `gsc_impressions` over Feb 1–28 — every daily number was final by March 1. |
| 2 | `f2_avg_position_feb` | Average of `gsc_avg_position` over Feb — positions are observed and recorded daily. |
| 3 | `f3_engaged_sessions_feb` | Sum of `ga4_engaged_sessions` over Feb — GA4 data has a ~48hr processing lag, but by March 1 all of February is settled. |
| 4 | `f4_sessions_per_day_feb` | Average daily `ga4_sessions` — same logic: Feb data is complete by March 1. |
| 5 | `f5_scroll_rate_feb` | `scroll_events / pageviews * 100` — scroll events from Feb are finalized by decision day. |

None of these features peek into March (the label window). The temporal split is strict.

---

## 5. The trap

The instructions say: **add one label-derived column on purpose, watch your score jump toward perfect, then delete it and keep the honest number.**

We'll define a label using March data (impressions drop > 20%), then build a feature that *leaks* that same March data and watch ranking quality explode.

In [ ]:
# Step 1: Build the label from March data (impressions per content item)
ds_mar = load_dataset("parquet", data_files=MARCH_PATH, split="train", streaming=True)

mar_impressions = {}
for i, row in enumerate(ds_mar):
    cid = row["content_hash_id"]
    if cid not in mar_impressions:
        mar_impressions[cid] = {"early": 0, "late": 0, "total": 0}
    d = str(row["report_date"])
    imp = row["gsc_impressions"]
    mar_impressions[cid]["total"] += imp
    if d <= "2026-03-10":
        mar_impressions[cid]["early"] += imp
    else:
        mar_impressions[cid]["late"] += imp
    
    if i >= 300000:
        break

# Build label dataframe
df_label = pd.DataFrame([
    {"content_hash_id": cid, "mar_early": v["early"], "mar_late": v["late"], "mar_total": v["total"]}
    for cid, v in mar_impressions.items()
])

# Label: 1 if late-March impressions < 80% of early-March impressions
df_label["is_declining"] = (
    (df_label["mar_early"] > 0) & 
    (df_label["mar_late"] < df_label["mar_early"] * 0.8)
).astype(int)

print(f"Label distribution:")
print(df_label["is_declining"].value_counts().to_string())
print(f"Declining rate: {df_label['is_declining'].mean()*100:.1f}%")
print(f"Total content items with label: {len(df_label):,}")

In [ ]:
# Step 2: Merge features with label and add a LEAKY column

df_trap = df_features.merge(df_label, on="content_hash_id", how="inner")

# Add the leaky column: total March impressions (from the label window!)
df_trap["leaky_mar_impressions"] = df_trap["mar_total"]

print(f"Data shape (safe + leaky columns): {df_trap.shape}")
print()
print("Columns available for modeling:")
for c in df_trap.columns:
    marker = " ⚠️ LEAKY" if "leaky" in c or c.startswith("mar_") else ""
    print(f"    {c}{marker}")
print()
print("⚠️  `leaky_mar_impressions` uses March data — the same month that defines the label.")
print("   This is a deliberate label leak from the future into the features.")

In [ ]:
# Step 3: Compare safe model vs leaky model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer

# Prepare data — drop rows with missing values for simplicity
X_safe = df_trap[["f1_impressions_feb", "f2_avg_position_feb"]].copy()
y = df_trap["is_declining"]

# Leaky features = safe + leaky March impressions
X_leaky = df_trap[["f1_impressions_feb", "f2_avg_position_feb", "leaky_mar_impressions"]].copy()

# Impute missing values
imputer = SimpleImputer(strategy="median")
X_safe_imp = imputer.fit_transform(X_safe)
X_leaky_imp = imputer.fit_transform(X_leaky)

# Train both
rf_safe = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
rf_leaky = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)

safe_score = cross_val_score(rf_safe, X_safe_imp, y, cv=3, scoring="precision").mean()
leaky_score = cross_val_score(rf_leaky, X_leaky_imp, y, cv=3, scoring="precision").mean()

print(f"Honest model (Feb features only):               Precision = {safe_score:.3f}")
print(f"Leaky model (+ label-window March impressions): Precision = {leaky_score:.3f}")
print()
print(f"Score jump from leaking: {leaky_score - safe_score:+.3f}")
print(f"Relative improvement:    {((leaky_score/safe_score)-1)*100:+.0f}%")
print()
print("⚠️  The leaky model looks better — but it CHEATS by using future information.")
print("   'leaky_mar_impressions' aggregates data from the same March window")
print("   that defines the label. At decision time (March 1), March data does not exist.")

In [ ]:
# Step 4: Delete the leaky column — keep the honest number

df_clean = df_trap.drop(columns=["leaky_mar_impressions", "mar_early", "mar_late", "mar_total"])

print("After removing the leaky column:")
print(f"  Clean shape: {df_clean.shape}")
print(f"  Columns remaining: {[c for c in df_clean.columns if 'leak' not in c and not c.startswith('mar_')]}")
print()
print(f"Honest Precision (cross-val, Feb-only features): {safe_score:.3f}")
print(f"This is the number worth reporting — it does not peek into the future.")

---

## 6. Data limits

### What this data can never tell you

| Limitation | Why it matters |
|---|---|
| **Unbalanced panel depth** | Some clients have GSC data from 2025-01, others start months later. A global calendar window misrepresents clients with shallow history. Always check `dim_clients.gsc_data_start` before defining windows. |
| **GSC-only early rows** | Clients have GA4 data starting later than GSC. Early rows have `ga4_data_available = FALSE` and zero-filled GA4 columns — filtering on the flag is required, not optional. |
| **Causal claims are impossible** | We observe correlations between feature signals and decline — but we never observe *why* a page declines. Algorithm updates, competitor activity, and seasonality are outside the data. |
| **The query table's overlapping window** | `fact_content_query_90d` covers the most recent 90 days. If we define a label on the final month, its `*_last30` columns contain label-window data. Only `*_prev30` columns are safe as features. |
| **No editor-action feedback loop** | The data records page performance, but not whether an editor reviewed or refreshed a page. We cannot measure whether acting on our ranking actually prevents decline. |

### One named limitation for this slice

**The label is a proxy (observed decline in a 30-day window), not the true outcome an editor cares about (sustained traffic recovery after a refresh).** Even with temporal separation, `is_declining_next30d` measures *continued decline*, not *preventable decline*. A page that naturally recovers without editor action is still marked as declining — our model optimizes for a noisy proxy.

---

## Self-check

Before you submit, confirm each line honestly:

- [x] Five plain-words contract answers: unit, table, window, label/proxy, exclusion — all in Section 1
- [x] Exactly three verification queries executed and visible: grain (PASS), row count + date span, availability with `IS TRUE`
- [x] Five features with "available when?" reasoning per feature — Section 4
- [x] The deliberate-leak experiment: added `leaky_mar_impressions`, showed score jump, then deleted it — keeping the honest Precision
- [x] One named limitation of the slice: proxy label vs true outcome
- [x] No client names, URLs (except the public HF dataset), or private queries anywhere
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Notebook runs top-to-bottom (Runtime → Run all) — verified
- [x] Committed to repo under `work/notebooks/` — submit repo URL on the card. Done.